In [1]:
import json, cv2
import numpy as np
from pathlib import Path
from collections import defaultdict
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle

REPO_ROOT = Path("../..").resolve()
NOMBRE = "A1606b0e6_0_62"
VIDEO_IN = REPO_ROOT / "data" / "raw" / "A1606b0e6_0 (62).mp4"
JSON_IN = REPO_ROOT / "data" / "tracking" / f"integrado_{NOMBRE}.json"
OUT_DIR = REPO_ROOT / "outputs" / "tactico_video"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(JSON_IN, encoding="utf-8") as fh:
    data = json.load(fh)
frames = data["frames"]
N_FRAMES = data["n_frames"]
color_equipo_bgr = {int(k): tuple(int(v) for v in val) for k, val in data["color_equipo_bgr"].items()}
col_eq_rgb = {eq: tuple(np.array(bgr[::-1]) / 255.0) for eq, bgr in color_equipo_bgr.items()}

MARGEN = 5.0
def en_campo(c):
    return c is not None and -MARGEN <= c[0] <= 105 + MARGEN and -MARGEN <= c[1] <= 68 + MARGEN

# Dirección de ataque por equipo (verificada): A ataca derecha, B ataca izquierda
ATAQUE = {0: "derecha", 1: "izquierda"}
PORTERIA_PROPIA = {0: 0.0, 1: 105.0}   # A defiende X=0, B defiende X=105

# Jugadores estables por equipo (excluye fragmentos)
MIN_DET = 200
det_id = defaultdict(int)
for fr in frames:
    for d in fr:
        if d.get("rol") in ("jugador", "portero") and d.get("equipo") in (0, 1) and en_campo(d.get("campo")):
            det_id[(d["equipo"], d["id"])] += 1
ids_validos = {(e, i) for (e, i), n in det_id.items() if n >= MIN_DET}

def jugadores_en_frame(f_idx, eq):
    """Jugadores de campo estables de un equipo en un frame, como lista de (x,y)."""
    return [d["campo"] for d in frames[f_idx]
            if d.get("rol") == "jugador" and d.get("equipo") == eq
            and en_campo(d.get("campo")) and (eq, d["id"]) in ids_validos]

print(f"Cargado. Equipo A ataca {ATAQUE[0]}, equipo B ataca {ATAQUE[1]}")
print(f"Jugadores estables: A={sum(1 for e,i in ids_validos if e==0)}, B={sum(1 for e,i in ids_validos if e==1)}")

Cargado. Equipo A ataca derecha, equipo B ataca izquierda
Jugadores estables: A=10, B=10


In [2]:
# ---- Asignación FIJA de líneas por formación (calcular una vez) ----
from sklearn.cluster import KMeans

CATALOGO = {
    "4-4-2":[4,4,2], "4-3-3":[4,3,3], "4-2-3-1":[4,2,3,1], "4-5-1":[4,5,1],
    "3-5-2":[3,5,2], "3-4-3":[3,4,3], "5-3-2":[5,3,2], "4-4-1-1":[4,4,1,1],
    "4-1-4-1":[4,1,4,1], "5-4-1":[5,4,1],
}

def asignar_lineas_fijas(eq):
    det = defaultdict(list)
    for fr in frames:
        for d in fr:
            if d.get("rol") == "jugador" and d.get("equipo") == eq and en_campo(d.get("campo")):
                det[d["id"]].append(d["campo"])
    jug = {i: np.median(ps, axis=0) for i, ps in det.items() if len(ps) >= MIN_DET}
    if len(jug) < 8:
        return None, None, None
    ids = list(jug)
    P = np.array([jug[i] for i in ids])
    avance = P[:, 0] if ATAQUE[eq] == "derecha" else (105 - P[:, 0])
    orden_av = np.argsort(avance)
    n = len(jug)
    mejor, mejor_coste = None, 1e9
    for nombre, patron in CATALOGO.items():
        if sum(patron) != n:
            continue
        coste, idx, centros = 0, 0, []
        for cuantos in patron:
            g = orden_av[idx:idx+cuantos]
            centros.append(avance[g].mean())
            coste += avance[g].std() * cuantos
            idx += cuantos
        if any(centros[i] >= centros[i+1] for i in range(len(centros)-1)):
            coste += 50
        if coste < mejor_coste:
            mejor_coste, mejor = coste, (nombre, patron)
    if mejor is None:
        return None, None, None
    nombre_form, patron = mejor
    linea_de, idx = {}, 0
    for li, cuantos in enumerate(patron):
        for j in orden_av[idx:idx+cuantos]:
            linea_de[ids[j]] = li
        idx += cuantos
    return linea_de, len(patron), nombre_form

lineas_fijas = {}
for eq in (0, 1):
    linea_de, n_lin, nombre = asignar_lineas_fijas(eq)
    lineas_fijas[eq] = (linea_de, n_lin, nombre)
    print(f"Equipo {'AB'[eq]}: formación fija 1-{nombre}" if nombre else f"Equipo {'AB'[eq]}: insuficiente")

Equipo A: formación fija 1-5-3-2
Equipo B: formación fija 1-3-4-3


In [4]:
# ---- Bases (pizarra cenital / partido perspectiva) + óvalos/triángulo + generador ----
from sklearn.cluster import KMeans

_npz = np.load(REPO_ROOT / "data" / "tracking" / f"homografias_pnl_{NOMBRE}.npz")
H_IMG2CAMPO = _npz["homografias"]
H_CAMPO2IMG = {}
for i in range(len(H_IMG2CAMPO)):
    Hi = H_IMG2CAMPO[i]
    if not np.isnan(Hi).any():
        try: H_CAMPO2IMG[i] = np.linalg.inv(Hi)
        except np.linalg.LinAlgError: pass

def campo_a_img(f_idx, x, y):
    H = H_CAMPO2IMG.get(f_idx)
    if H is None: return None
    p = cv2.perspectiveTransform(np.array([[[x, y]]], dtype=np.float32), H)[0][0]
    return (int(p[0]), int(p[1]))

def _suaviza_serie(serie, VENTANA, MAX_HUECO=15, MAX_VEL=1.2):
    if len(serie) < 2: return dict(serie)
    fs = sorted(serie); out = {fs[0]: serie[fs[0]]}; fa, pa = fs[0], serie[fs[0]]
    for f in fs[1:]:
        p = serie[f]
        if np.hypot(p[0]-pa[0], p[1]-pa[1])/max(f-fa,1) <= MAX_VEL:
            out[f] = p; fa, pa = f, p
    fs2 = sorted(out); comp = {}
    for a,b in zip(fs2, fs2[1:]):
        comp[a]=out[a]; h=b-a
        if 1<h<=MAX_HUECO:
            xa,ya=out[a]; xb,yb=out[b]
            for k in range(1,h): comp[a+k]=(xa+(k/h)*(xb-xa), ya+(k/h)*(yb-ya))
    comp[fs2[-1]]=out[fs2[-1]]
    fs3=sorted(comp); arr=np.array([comp[f] for f in fs3]); sm={}; hh=VENTANA//2
    for i,f in enumerate(fs3):
        sm[f]=tuple(arr[max(0,i-hh):i+hh+1].mean(axis=0))
    return sm

def trayectorias_lineas(VENTANA=21):
    tray = defaultdict(dict)
    for f_idx, fr in enumerate(frames):
        for d in fr:
            if d.get("rol") != "jugador" or d.get("equipo") not in (0,1) or not en_campo(d.get("campo")):
                continue
            tray[(d["equipo"], d["id"])][f_idx] = tuple(d["campo"])
    return {k: _suaviza_serie(s, VENTANA) for k,s in tray.items()}

TRAY_LIN = trayectorias_lineas(VENTANA=21)

def trayectorias_suaves(VENTANA=5):
    tray = defaultdict(dict); meta = {}
    for f_idx, fr in enumerate(frames):
        for d in fr:
            rol = d.get("rol")
            if rol not in ("jugador","portero","arbitro","balon") or not en_campo(d.get("campo")):
                continue
            tray[(rol, d["id"])][f_idx] = tuple(d["campo"])
            meta[(rol, d["id"])] = {"rol": rol, "equipo": d.get("equipo", -1)}
    return {k: _suaviza_serie(s, VENTANA) for k,s in tray.items()}, meta

TRAY, META = trayectorias_suaves(VENTANA=5)

def pos_en_frame(f_idx):
    jug, arb, bal = [], [], None
    for key, serie in TRAY.items():
        if f_idx not in serie: continue
        rol, _id = key; x, y = serie[f_idx]; eq = META[key]["equipo"]
        if rol in ("jugador","portero"): jug.append((x,y,eq,_id))
        elif rol == "arbitro": arb.append((x,y))
        elif rol == "balon": bal = (x,y)
    return jug, arb, bal

# --- Helpers de dibujo: óvalo HORIZONTAL para jugadores/árbitros, triángulo-flecha para el balón ---
def dibujar_jugador(img, cx, cy, color, hueco=False, escala=1.0):
    ejes = (int(13*escala), int(9*escala))   # óvalo HORIZONTAL (ancho > alto)
    if hueco:
        cv2.ellipse(img, (cx, cy), ejes, 0, 0, 360, color, 4)
    else:
        cv2.ellipse(img, (cx, cy), ejes, 0, 0, 360, color, -1)
        cv2.ellipse(img, (cx, cy), ejes, 0, 0, 360, (255,255,255), 2)

def dibujar_balon(img, cx, cy, escala=1.0):
    s = int(7*escala); top = cy - int(18*escala)   # triángulo más pequeño
    pts = np.array([[cx-s, top], [cx+s, top], [cx, top+int(11*escala)]], dtype=np.int32)
    cv2.fillPoly(img, [pts], (0, 255, 255))
    cv2.polylines(img, [pts], True, (0, 0, 0), 1)

# --- BASE 1: pizarra esquemática (cenital) ---
def base_pizarra(f_idx, W=1050, H=680):
    img = np.zeros((H, W, 3), dtype=np.uint8); sx, sy = W/105.0, H/68.0
    def px(x,y): return (int(x*sx), int(y*sy))
    img[:] = (45,95,50); bl=(255,255,255)
    cv2.rectangle(img, px(0,0), px(105,68), bl, 2); cv2.line(img, px(52.5,0), px(52.5,68), bl, 2)
    cv2.circle(img, px(52.5,34), int(9.15*sx), bl, 2)
    cv2.rectangle(img, px(0,13.84), px(16.5,54.16), bl, 2); cv2.rectangle(img, px(88.5,13.84), px(105,54.16), bl, 2)
    jug, arb, bal = pos_en_frame(f_idx)
    for (x,y,eq,_id) in jug:
        c = color_equipo_bgr.get(eq,(128,128,128))
        dibujar_jugador(img, *px(x,y), c, hueco=False, escala=1.0)
    for (x,y) in arb:
        dibujar_jugador(img, *px(x,y), (30,30,30), hueco=False, escala=0.9)
    if bal: dibujar_balon(img, *px(*bal), escala=1.0)
    return img, ("cenital", px)

# --- BASE 2: partido (perspectiva), óvalos huecos grandes + triángulo balón ---
_cap = {"c": None}
def base_partido(f_idx):
    if _cap["c"] is None: _cap["c"] = cv2.VideoCapture(str(VIDEO_IN))
    _cap["c"].set(cv2.CAP_PROP_POS_FRAMES, f_idx)
    ret, frame = _cap["c"].read()
    if not ret: frame = np.zeros((1080,1920,3), dtype=np.uint8)
    for d in frames[f_idx]:
        rol = d.get("rol")
        if "bbox" not in d: continue
        x1,y1,x2,y2 = [int(v) for v in d["bbox"]]
        cx, cy = (x1+x2)//2, y2
        if rol in ("jugador","portero"):
            c = color_equipo_bgr.get(d.get("equipo"), (128,128,128))
            dibujar_jugador(frame, cx, cy, c, hueco=True, escala=1.4)
        elif rol == "arbitro":
            dibujar_jugador(frame, cx, cy, (30,30,30), hueco=True, escala=1.2)
        elif rol == "balon":
            dibujar_balon(frame, cx, cy, escala=1.3)
    return frame, ("perspectiva", f_idx)

# --- CAPA líneas ---
def capa_lineas(img, ctx, f_idx):
    modo = ctx[0]
    for eq in (0,1):
        linea_de, n_lin, _ = lineas_fijas[eq]
        if linea_de is None: continue
        base = color_equipo_bgr[eq]
        por_linea = defaultdict(list)
        for (e,_id), serie in TRAY_LIN.items():
            if e != eq or _id not in linea_de: continue
            if f_idx in serie: por_linea[linea_de[_id]].append(serie[f_idx])
        for li in range(n_lin):
            pts_m = sorted(por_linea.get(li,[]), key=lambda p:p[1])
            if modo == "cenital":
                px = ctx[1]; pts = [px(x,y) for (x,y) in pts_m]
            else:
                pts = [campo_a_img(f_idx, x, y) for (x,y) in pts_m]; pts = [p for p in pts if p]
            for k in range(len(pts)-1):
                cv2.line(img, pts[k], pts[k+1], base, 3)
            for p in pts: cv2.circle(img, p, 4, base, -1)

def generar(nombre, capa_fn, base="pizarra"):
    img0, _ = (base_pizarra(0) if base=="pizarra" else base_partido(0)); H,W = img0.shape[:2]
    out = cv2.VideoWriter(str(OUT_DIR / f"{nombre}_{base}_{NOMBRE}.mp4"),
                          cv2.VideoWriter_fourcc(*"mp4v"), 25, (W,H))
    for f_idx in range(N_FRAMES):
        img, ctx = (base_pizarra(f_idx) if base=="pizarra" else base_partido(f_idx))
        capa_fn(img, ctx, f_idx)
        out.write(img)
    out.release(); print(f"Generado: {nombre}_{base}_{NOMBRE}.mp4")

generar("lineas", capa_lineas, base="pizarra")
generar("lineas", capa_lineas, base="partido")

Generado: lineas_pizarra_A1606b0e6_0_62.mp4
Generado: lineas_partido_A1606b0e6_0_62.mp4


In [4]:
# ---- Capas: centroide, dispersión, posesión (NEUTRO relajado). Ambas bases ----
def a_pixel(ctx, f_idx, x, y):
    if ctx[0] == "cenital": return ctx[1](x, y)
    return campo_a_img(f_idx, x, y)

def capa_centroide(img, ctx, f_idx):
    for eq in (0, 1):
        pts = [serie[f_idx] for (e,_id), serie in TRAY_LIN.items() if e==eq and f_idx in serie]
        if len(pts) < 3: continue
        cen = np.array(pts).mean(axis=0); c = color_equipo_bgr[eq]
        p = a_pixel(ctx, f_idx, cen[0], cen[1])
        if p is None: continue
        cv2.circle(img, p, 14, c, -1); cv2.circle(img, p, 14, (255,255,255), 2); cv2.circle(img, p, 20, c, 2)

def capa_dispersion(img, ctx, f_idx):
    for eq in (0, 1):
        pts = np.array([serie[f_idx] for (e,_id), serie in TRAY_LIN.items() if e==eq and f_idx in serie])
        if len(pts) < 3: continue
        cen = pts.mean(axis=0); c = color_equipo_bgr[eq]
        radio_m = np.linalg.norm(pts - cen, axis=1).std() * 2
        perim = []
        for ang in np.linspace(0, 2*np.pi, 40):
            p = a_pixel(ctx, f_idx, cen[0]+radio_m*np.cos(ang), cen[1]+radio_m*np.sin(ang))
            if p: perim.append(p)
        if len(perim) > 3:
            cv2.polylines(img, [np.array(perim, dtype=np.int32)], True, c, 2)
        pc = a_pixel(ctx, f_idx, cen[0], cen[1])
        if pc: cv2.circle(img, pc, 5, c, -1)

# Posesión por frame con NEUTRO RELAJADO (dist_control=10, franja_disputa=1.0)
posesion_frame = {}
for f_idx in range(N_FRAMES):
    bpos = next((d["campo"] for d in frames[f_idx] if d.get("rol")=="balon" and en_campo(d.get("campo"))), None)
    if bpos is None:
        posesion_frame[f_idx] = None; continue
    dmin = {0:999, 1:999}
    for d in frames[f_idx]:
        if d.get("rol")=="jugador" and d.get("equipo") in (0,1) and en_campo(d.get("campo")):
            dd = np.hypot(d["campo"][0]-bpos[0], d["campo"][1]-bpos[1])
            dmin[d["equipo"]] = min(dmin[d["equipo"]], dd)
    d0,d1 = dmin[0], dmin[1]
    if min(d0,d1) > 10 or abs(d0-d1) < 1.0:
        posesion_frame[f_idx] = "neutro"
    else:
        posesion_frame[f_idx] = 0 if d0 < d1 else 1

_acum_pos = []
_a = {0:0, 1:0, "neutro":0}
for f_idx in range(N_FRAMES):
    e = posesion_frame.get(f_idx)
    if e in (0,1,"neutro"): _a[e] += 1
    _acum_pos.append((_a[0], _a[1], _a["neutro"]))

COLOR_NEUTRO = (160, 160, 160)
def capa_posesion(img, ctx, f_idx):
    H, W = img.shape[:2]
    a0, a1, an = _acum_pos[f_idx]
    tot = a0 + a1 + an
    bar_w = int(W*0.6); x0 = (W-bar_w)//2; y0 = 20; h = 22
    if tot > 0:
        w0 = int(bar_w*a0/tot); wn = int(bar_w*an/tot)
        cv2.rectangle(img, (x0,y0), (x0+w0,y0+h), color_equipo_bgr[0], -1)
        cv2.rectangle(img, (x0+w0,y0), (x0+w0+wn,y0+h), COLOR_NEUTRO, -1)
        cv2.rectangle(img, (x0+w0+wn,y0), (x0+bar_w,y0+h), color_equipo_bgr[1], -1)
        cv2.rectangle(img, (x0,y0), (x0+bar_w,y0+h), (255,255,255), 2)
        cv2.putText(img, f"{100*a0/tot:.0f}%", (x0+5,y0+17), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
        cv2.putText(img, f"N{100*an/tot:.0f}%", (x0+w0+3,y0+16), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0,0,0), 1)
        cv2.putText(img, f"{100*a1/tot:.0f}%", (x0+bar_w-48,y0+17), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,0), 2)
    estado = posesion_frame.get(f_idx)
    bpos = next((d["campo"] for d in frames[f_idx] if d.get("rol")=="balon" and en_campo(d.get("campo"))), None)
    if bpos and estado is not None:
        p = a_pixel(ctx, f_idx, bpos[0], bpos[1])
        if p:
            col = COLOR_NEUTRO if estado == "neutro" else color_equipo_bgr[estado]
            cv2.circle(img, p, 16, col, 3)

for base in ("pizarra", "partido"):
    generar("centroide", capa_centroide, base=base)
    generar("dispersion", capa_dispersion, base=base)
    generar("posesion", capa_posesion, base=base)

Generado: centroide_pizarra_A1606b0e6_0_62.mp4
Generado: dispersion_pizarra_A1606b0e6_0_62.mp4
Generado: posesion_pizarra_A1606b0e6_0_62.mp4
Generado: centroide_partido_A1606b0e6_0_62.mp4
Generado: dispersion_partido_A1606b0e6_0_62.mp4
Generado: posesion_partido_A1606b0e6_0_62.mp4


In [5]:
# ---- Mapa de calor ANIMADO, dos paneles (un equipo cada uno), acumulación progresiva ----
from scipy.ndimage import gaussian_filter
import matplotlib.cm as cm

def render_calor_animado(cada=3):
    """Vídeo de 2 paneles. El calor se acumula frame a frame; al final, el mapa completo."""
    BINS_X, BINS_Y = 52, 34
    grids = {0: np.zeros((BINS_Y, BINS_X)), 1: np.zeros((BINS_Y, BINS_X))}
    # tamaño de cada panel y lienzo (dos campos lado a lado)
    PW, PH = 525, 340
    W, H = PW*2 + 30, PH + 60
    out = cv2.VideoWriter(str(OUT_DIR / f"calor_pizarra_{NOMBRE}.mp4"),
                          cv2.VideoWriter_fourcc(*"mp4v"), 25, (W, H))
    cmap = cm.get_cmap("hot")

    def panel_equipo(eq):
        grid = grids[eq]
        heat = gaussian_filter(grid, sigma=1.5)
        img = np.zeros((PH, PW, 3), dtype=np.uint8); img[:] = (26, 58, 23)
        if heat.max() > 0:
            norm = (heat / heat.max())
            # colorear con colormap hot y mezclar sobre el campo
            colored = (cmap(norm)[:, :, :3] * 255).astype(np.uint8)  # RGB
            colored = cv2.cvtColor(colored, cv2.COLOR_RGB2BGR)
            colored = cv2.resize(colored, (PW, PH), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize((norm*255).astype(np.uint8), (PW, PH))
            alpha = (mask.astype(float)/255*0.7)[:,:,None]
            img = (img*(1-alpha) + colored*alpha).astype(np.uint8)
        # líneas del campo
        sx, sy = PW/105.0, PH/68.0
        def px(x,y): return (int(x*sx), int(y*sy))
        bl=(255,255,255)
        cv2.rectangle(img, px(0,0), px(104,67), bl, 1); cv2.line(img, px(52.5,0), px(52.5,68), bl, 1)
        cv2.circle(img, px(52.5,34), int(9.15*sx), bl, 1)
        cv2.rectangle(img, px(0,13.84), px(16.5,54.16), bl, 1); cv2.rectangle(img, px(88.5,13.84), px(105,54.16), bl, 1)
        return img

    for f_idx in range(N_FRAMES):
        # acumular posiciones de este frame
        for d in frames[f_idx]:
            if d.get("rol")=="jugador" and d.get("equipo") in (0,1) and en_campo(d.get("campo")):
                x,y = d["campo"]
                xi = min(max(int(x/105*BINS_X),0),BINS_X-1); yi = min(max(int(y/68*BINS_Y),0),BINS_Y-1)
                grids[d["equipo"]][yi,xi] += 1
        if f_idx % cada != 0 and f_idx != N_FRAMES-1:
            continue
        lienzo = np.zeros((H, W, 3), dtype=np.uint8); lienzo[:] = (20,20,20)
        for eq in (0,1):
            pan = panel_equipo(eq)
            x0 = 10 + eq*(PW+10)
            lienzo[40:40+PH, x0:x0+PW] = pan
            col = color_equipo_bgr[eq]
            cv2.putText(lienzo, f"Equipo {'AB'[eq]}", (x0+10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
        # repetir frame 'cada' veces para mantener fps
        for _ in range(cada):
            out.write(lienzo)
    out.release(); print(f"Generado: calor_pizarra_{NOMBRE}.mp4")

render_calor_animado(cada=3)

C:\Users\sgghu\AppData\Local\Temp\ipykernel_35344\2882232178.py:14: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = cm.get_cmap("hot")


Generado: calor_pizarra_A1606b0e6_0_62.mp4
